# H/k-Prototyp fuer GitLab Task 44

Dieses Notebook ist der zentrale Testpunkt fuer den ersten Architektur-Prototypen:

- Planner erzeugt Subgoals und erwartete Outcomes.
- Executor fuehrt die Subgoals deterministisch in BrowserGym aus.
- Evaluator prueft Zwischenzustaende nach dem Intervall `k`.
- Controller entscheidet `continue`, `local_replan`, `global_replan` oder `abort`.
- WebArena-Verified bleibt die offizielle finale Bewertungsinstanz.

Wichtig: Demo-GitLab wird hier nicht automatisch gestartet. Wenn GitLab aus ist, zuerst im Terminal starten:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
RUNNER = PROJECT_ROOT / 'scripts' / 'run_hk_task44_prototype.py'
PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'

TASK_ID = 44
CONFIG = 'examples/configs/config.demo.json'
TASKS_FILE = 'output/tasks.demo.json'

PROJECT_ROOT, OFFICIAL_REPO, RUNNER

## 1. Task-Input sicherstellen

Diese Zelle erzeugt den Agent-Input fuer Task 44. Sie braucht keinen laufenden Browser, aber das WebArena-Verified-Repo muss eingerichtet sein.

In [ ]:
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', str(TASK_ID),
    '--config', CONFIG,
    '--output', TASKS_FILE,
], cwd=OFFICIAL_REPO, check=True)

tasks_path = OFFICIAL_REPO / TASKS_FILE
json.loads(tasks_path.read_text())

## 2. Einzelnen H/k-Lauf ausfuehren

Falls Demo-GitLab gestoppt ist, schlaegt diese Zelle erwartbar fehl. Dann GitLab im Terminal starten und die Zelle erneut ausfuehren.

In [ ]:
H = 0
K = 1

subprocess.run([
    str(PYTHON), str(RUNNER),
    '--repo-root', str(OFFICIAL_REPO),
    '--tasks-file', TASKS_FILE,
    '--task-id', str(TASK_ID),
    '--config', CONFIG,
    '--h', str(H),
    '--k', str(K),
], cwd=PROJECT_ROOT, check=True)

## 3. Artefakte laden

In [ ]:
run_dir = OFFICIAL_REPO / 'output' / 'hk-prototype' / f'h{H}_k{K}' / str(TASK_ID)

summary = json.loads((run_dir / 'run_summary.json').read_text())
plan = json.loads((run_dir / 'plan.json').read_text())
eval_result = json.loads((run_dir / 'eval_result.json').read_text())

summary

In [ ]:
plan

## 4. Trace, Evaluator-Signale und Controller-Entscheidungen anzeigen

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.DataFrame(json.loads(line) for line in path.read_text().splitlines())

step_trace = read_jsonl(run_dir / 'step_trace.jsonl')
evaluator_signals = read_jsonl(run_dir / 'evaluator_signals.jsonl')
controller_decisions = read_jsonl(run_dir / 'controller_decisions.jsonl')

step_trace

In [ ]:
evaluator_signals

In [ ]:
controller_decisions

## 5. Kleine H/k-Grid-Auswertung

Diese Zelle kann mehrere Konfigurationen laufen lassen. Fuer Task 44 ist der Executor deterministisch; Unterschiede sieht man erstmal vor allem in Evaluationsfrequenz und Logs. Spaeter wird hier der LLM-Planner bzw. ein semantischer Evaluator interessant.

In [ ]:
RUN_GRID = False
GRID = [(0, 1), (0, 2), (2, 1), (2, 2)]

if RUN_GRID:
    for h, k in GRID:
        subprocess.run([
            str(PYTHON), str(RUNNER),
            '--repo-root', str(OFFICIAL_REPO),
            '--tasks-file', TASKS_FILE,
            '--task-id', str(TASK_ID),
            '--config', CONFIG,
            '--h', str(h),
            '--k', str(k),
        ], cwd=PROJECT_ROOT, check=True)

rows = []
for summary_path in (OFFICIAL_REPO / 'output' / 'hk-prototype').glob('h*_k*/44/run_summary.json'):
    rows.append(json.loads(summary_path.read_text()))

summary_df = pd.DataFrame(rows)
summary_df[['task_id', 'h', 'k', 'planner_mode', 'total_steps', 'total_runtime_ms', 'success', 'score', 'final_url']] if not summary_df.empty else summary_df

## 6. Einordnung fuer den naechsten Ausbau

Der offizielle WebArena-Evaluator bleibt die finale Benchmark-Bewertung. Der interne Evaluator prueft dagegen Laufzeit- und Zwischenzustaende. Spaeter kann er hybrid werden:

- programmatische Checks: URL, Login-Zustand, bekannte DOM-Elemente, HAR-Ereignisse
- heuristische Checks: No-Progress, Loops, wiederholte URLs, Invalid Actions
- optionaler LLM-Judge: semantische Pruefung, ob Observation und Subgoal sinnvoll zusammenpassen

So bleibt die finale Bewertung reproduzierbar, waehrend der Agent zur Laufzeit trotzdem bessere Kontextpruefungen bekommt.